In [5]:
# =============================================================================
# -*- coding: utf-8 -*-
"""
IROS Column Guard + Active-Only Export (T0-safe, no source overwrite) — v2

Fix vs your run:
- Phase 2A adds debug columns (is_transition, label_state, onset_score, offset_score,
  p_action_mean, w_label, w_trial).
- Old Phase 2B treated these as "extras" => skipped every file.
- This v2 explicitly allows those columns and (optionally) keeps them at the end.

What it does
- Scans:   ROOT_DIR/Sub-*/cleaned/synchronized_proper_lite_union_v3/label/*_icml_consensus_labels.csv
- Validates schema against allowed set:
    canonical + optional + Phase2A debug extras
- Reorders into canonical header (does NOT modify source files)
- Exports into: .../synchronized_proper_lite_union_v3/labelonly/<same filename>
- Non-T0: filters rows to active==1
- T0: bypass filtering and exports full (reordered) file
- Optionally copies JSON sidecars
- Writes logs:
    column_invalid_or_skipped.csv
    active_zero_rows_removed.csv
"""

from __future__ import annotations

import csv
import glob
import re
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd


# ----------------------------- CONFIG -----------------------------
ROOT_DIR = Path(r"/home/tsultan1/IROS/Data")
SRC_LABEL_SUBPATH = Path(r"cleaned/synchronized_proper_lite_union_v3/label")
DEST_LABEL_SUBNAME = "labelonly"
CSV_GLOB = "*_icml_consensus_labels.csv"

COPY_SIDECARS = True
DELETE_IF_EMPTY = False               # only affects DEST output
OVERWRITE_DEST = False                # True => overwrite destination CSVs if they exist

# If True: require *all* CANONICAL_BASE columns to exist in the source label CSV.
# If False: require only minimal label + meta columns; modality columns can be missing.
REQUIRE_STRICT_SCHEMA = True

# If True: keep Phase-2A debug columns at the end of the output CSV (useful for analysis).
# If False: drop them (cleaner for training).
KEEP_DEBUG_COLS = False

# Canonical header order (matches your “final synced + labels” expectation)
CANONICAL_BASE = [
    "Timestamp_seconds",
    # (optional 'Timestamp_ms' will be inserted here if present)
    "EMG_Ch1", "EMG_Ch2", "EMG_Ch3", "EMG_Ch4",
    "EEG_Ch1", "EEG_Ch2", "EEG_Ch3", "EEG_Ch4", "EEG_Ch5", "EEG_Ch6", "EEG_Ch7", "EEG_Ch8",
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
    "ET_Blink", "ET_Fixation", "ET_Worn",
    # (optional ET distances will be inserted here if present)
    "ET_GyroX", "ET_GyroY", "ET_GyroZ",
    "ET_AccX", "ET_AccY", "ET_AccZ",
    "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
    "active_raw", "active", "active_prob", "label_action",
    "subject_id", "task", "trial", "label_11", "task_target",
]

OPTIONAL_COLS = {
    "Timestamp_ms",
    "ET_DistanceLeft", "ET_DistanceRight",
}

# ✅ Phase-2A debug/aux columns that are OK to exist
PHASE2A_DEBUG_COLS = {
    "is_transition",
    "label_state",
    "onset_score",
    "offset_score",
    "p_action_mean",
    "w_label",
    "w_trial",
}

# Minimal required columns if REQUIRE_STRICT_SCHEMA = False
MIN_REQUIRED_COLS = {
    "Timestamp_seconds",
    "active_raw", "active", "active_prob", "label_action",
    "subject_id", "task", "trial", "label_11", "task_target",
}

LOG_INVALID_PATH = ROOT_DIR / "column_invalid_or_skipped.csv"
LOG_FILTER_PATH = ROOT_DIR / "active_zero_rows_removed.csv"


# ----------------------------- HELPERS -----------------------------
def is_subject_folder_name(folder_name: str) -> bool:
    return re.match(r"(?i)^sub-?\d+$", folder_name.strip()) is not None


def normalize_column_name(name: str) -> str:
    """Remove BOM, trim, and normalize whitespace."""
    s = (name or "").replace("\ufeff", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_header_list(header: list[str]) -> list[str]:
    return [normalize_column_name(h) for h in header]


def walk_label_csv_files() -> list[Path]:
    files: list[Path] = []
    for sub in sorted(d for d in ROOT_DIR.iterdir() if d.is_dir() and is_subject_folder_name(d.name)):
        src_label_dir = sub / SRC_LABEL_SUBPATH
        if not src_label_dir.exists():
            print(f"[skip] label folder missing: {src_label_dir}")
            continue
        hits = sorted(Path(p) for p in glob.glob(str(src_label_dir / CSV_GLOB)))
        files.extend(hits)
    return files


def build_canonical_order(existing_cols: list[str]) -> list[str]:
    """
    Build desired column order:
      - canonical base (+ Timestamp_ms and ET distances inserted if present)
      - optionally append debug cols (if KEEP_DEBUG_COLS)
    Keeps only columns that exist.
    """
    has_ts_ms = "Timestamp_ms" in existing_cols
    has_distL = "ET_DistanceLeft" in existing_cols
    has_distR = "ET_DistanceRight" in existing_cols

    target: list[str] = []
    for name in CANONICAL_BASE:
        if name == "Timestamp_seconds":
            target.append("Timestamp_seconds")
            if has_ts_ms:
                target.append("Timestamp_ms")
        elif name == "ET_Worn":
            target.append("ET_Worn")
            if has_distL:
                target.append("ET_DistanceLeft")
            if has_distR:
                target.append("ET_DistanceRight")
        else:
            target.append(name)

    target = [c for c in target if c in existing_cols]

    if KEEP_DEBUG_COLS:
        for c in sorted(PHASE2A_DEBUG_COLS):
            if c in existing_cols and c not in target:
                target.append(c)

    return target


def detect_rest_trial_T0(src_path: Path) -> bool:
    """
    Detect Task 0 from filename using rule:
      (T|M)(\d{2,}) and FIRST digit after T/M is task.
    Example: T01 -> task=0, trial=1
    """
    stem = src_path.stem
    m = re.search(r"(?:^|_)(T|M)(\d{2,})", stem, flags=re.I)
    if not m:
        return False
    kind = m.group(1).upper()
    if kind != "T":
        return False
    digits = m.group(2)
    task_digit = int(digits[0])  # first digit only
    return task_digit == 0


def make_destination_path(src_csv: Path) -> Path:
    src_label_dir = src_csv.parent
    dest_dir = src_label_dir.parent / DEST_LABEL_SUBNAME
    dest_dir.mkdir(parents=True, exist_ok=True)
    return dest_dir / src_csv.name


def list_sidecar_files(src_csv: Path) -> list[Path]:
    return [
        src_csv.with_name(src_csv.stem.replace("_icml_consensus_labels", "_onsets") + ".json"),
        src_csv.with_suffix(".unitconv.json"),
    ]


def read_csv_header_only(path: Path) -> list[str]:
    with open(path, "r", newline="", encoding="utf-8-sig") as fh:
        reader = csv.reader(fh)
        header_raw = next(reader)
    return normalize_header_list(header_raw)


def read_dataframe_normalized(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    df.columns = normalize_header_list([str(c) for c in df.columns])
    return df


def safe_active_mask(df: pd.DataFrame) -> pd.Series:
    """Robust active==1 mask even if active is float/string/NaN."""
    active_num = pd.to_numeric(df["active"], errors="coerce").fillna(0).astype("int64")
    return active_num == 1


@dataclass
class AuditRowInvalid:
    file: str
    when: str
    missing: str
    extra: str


@dataclass
class AuditRowFilter:
    file: str
    when: str
    rows_before: int
    rows_removed: int
    rows_after: int
    dest: str


# ----------------------------- MAIN RUNNER -----------------------------
def run_column_guard_and_export() -> None:
    allowed_set = set(CANONICAL_BASE) | set(OPTIONAL_COLS) | set(PHASE2A_DEBUG_COLS)
    required_set = set(CANONICAL_BASE) if REQUIRE_STRICT_SCHEMA else set(MIN_REQUIRED_COLS)

    invalid_log: list[dict] = []
    filter_log: list[dict] = []

    n_files = 0
    n_exported = 0
    n_reordered = 0
    n_filtered_files = 0
    n_t0_passthrough = 0
    n_invalid_or_skipped = 0
    n_skipped_dest_exists = 0

    all_files = walk_label_csv_files()
    print(f"[info] Found {len(all_files)} label CSVs")

    for src_csv in all_files:
        n_files += 1
        try:
            header = read_csv_header_only(src_csv)

            cur_set = set(header)
            extras = sorted(list(cur_set - allowed_set))
            missing_required = sorted(list(required_set - cur_set))

            if extras or missing_required:
                msg = f"invalid schema: missing={missing_required} extras={extras}"
                print(f"[skip-invalid] {src_csv} | {msg}")
                invalid_log.append(AuditRowInvalid(
                    file=str(src_csv),
                    when=datetime.now().isoformat(timespec="seconds"),
                    missing=";".join(missing_required),
                    extra=";".join(extras),
                ).__dict__)
                n_invalid_or_skipped += 1
                continue

            df = read_dataframe_normalized(src_csv)
            header_now = list(df.columns)

            # Ensure no unexpected extras after pandas read
            extras2 = sorted(list(set(header_now) - allowed_set))
            if extras2:
                print(f"[skip-invalid] {src_csv} | unexpected columns after read: {extras2}")
                invalid_log.append(AuditRowInvalid(
                    file=str(src_csv),
                    when=datetime.now().isoformat(timespec="seconds"),
                    missing="",
                    extra=";".join(extras2),
                ).__dict__)
                n_invalid_or_skipped += 1
                continue

            target_order = build_canonical_order(header_now)

            if header_now != target_order:
                df = df[target_order]
                n_reordered += 1

            dest_csv = make_destination_path(src_csv)

            if dest_csv.exists() and not OVERWRITE_DEST:
                print(f"[skip] dest exists (no-overwrite): {dest_csv}")
                n_skipped_dest_exists += 1
                continue

            # Copy sidecars first (optional)
            if COPY_SIDECARS:
                for sc in list_sidecar_files(src_csv):
                    if sc.exists():
                        dest_sc = dest_csv.parent / sc.name
                        try:
                            if (not dest_sc.exists()) or OVERWRITE_DEST:
                                shutil.copy2(sc, dest_sc)
                        except Exception as e:
                            print(f"  [warn] sidecar copy failed {sc} → {dest_sc}: {e}")

            # T0: no filtering
            if detect_rest_trial_T0(src_csv):
                print(f"[export T0-no-filter] {dest_csv.name}")
                df.to_csv(dest_csv, index=False)
                n_t0_passthrough += 1
                n_exported += 1
                continue

            # Non-T0: active-only
            if "active" not in df.columns:
                print(f"[skip-invalid] {src_csv} | missing 'active' column")
                invalid_log.append(AuditRowInvalid(
                    file=str(src_csv),
                    when=datetime.now().isoformat(timespec="seconds"),
                    missing="active",
                    extra="",
                ).__dict__)
                n_invalid_or_skipped += 1
                continue

            before = len(df)
            mask = safe_active_mask(df)
            df_f = df.loc[mask].copy()
            removed = before - len(df_f)

            if removed > 0:
                n_filtered_files += 1
                filter_log.append(AuditRowFilter(
                    file=str(src_csv),
                    when=datetime.now().isoformat(timespec="seconds"),
                    rows_before=before,
                    rows_removed=removed,
                    rows_after=len(df_f),
                    dest=str(dest_csv),
                ).__dict__)
                print(f"[filter] {src_csv.name} kept={len(df_f)} removed={removed}")

            if len(df_f) == 0 and DELETE_IF_EMPTY:
                print(f"[note] empty after filter → not exporting (DELETE_IF_EMPTY=True): {src_csv.name}")
                continue

            df_f.to_csv(dest_csv, index=False)
            n_exported += 1

        except StopIteration:
            print(f"[skip-invalid] {src_csv} | empty or no header")
            invalid_log.append(AuditRowInvalid(
                file=str(src_csv),
                when=datetime.now().isoformat(timespec="seconds"),
                missing="ALL",
                extra="",
            ).__dict__)
            n_invalid_or_skipped += 1
        except Exception as e:
            print(f"[ERROR] {src_csv} | {e}")
            invalid_log.append(AuditRowInvalid(
                file=str(src_csv),
                when=datetime.now().isoformat(timespec="seconds"),
                missing="EXCEPTION",
                extra=str(e),
            ).__dict__)
            n_invalid_or_skipped += 1

    # Write logs
    if invalid_log:
        pd.DataFrame(invalid_log).to_csv(LOG_INVALID_PATH, index=False)
        print(f"[log] Invalid/Skipped list → {LOG_INVALID_PATH}")

    if filter_log:
        pd.DataFrame(filter_log).to_csv(LOG_FILTER_PATH, index=False)
        print(f"[log] Active-row filter stats → {LOG_FILTER_PATH}")

    print(
        "\n[summary] "
        f"scanned={n_files}  exported={n_exported}  reordered={n_reordered}  "
        f"filtered_files={n_filtered_files}  T0_passthrough={n_t0_passthrough}  "
        f"dest_exists_skipped={n_skipped_dest_exists}  invalid_or_skipped={n_invalid_or_skipped}"
    )


# ----------------------------- ENTRYPOINT -----------------------------
if __name__ == "__main__":
    run_column_guard_and_export()


<>:167: SyntaxWarning: invalid escape sequence '\d'
<>:167: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_85652/570673413.py:167: SyntaxWarning: invalid escape sequence '\d'
  """


[info] Found 2487 label CSVs
[export T0-no-filter] 001_T03_synchronized_corrected_icml_consensus_labels.csv
[filter] 002_T516_synchronized_corrected_icml_consensus_labels.csv kept=1322 removed=1266
[filter] 003_T515_synchronized_corrected_icml_consensus_labels.csv kept=2006 removed=1172
[filter] 004_T514_synchronized_corrected_icml_consensus_labels.csv kept=1728 removed=1000
[filter] 005_T513_synchronized_corrected_icml_consensus_labels.csv kept=1215 removed=1596
[filter] 006_T512_synchronized_corrected_icml_consensus_labels.csv kept=1125 removed=2045
[filter] 007_T511_synchronized_corrected_icml_consensus_labels.csv kept=1270 removed=1841
[filter] 008_T510_synchronized_corrected_icml_consensus_labels.csv kept=1262 removed=1754
[filter] 009_T59_synchronized_corrected_icml_consensus_labels.csv kept=1322 removed=1000
[filter] 010_T416_synchronized_corrected_icml_consensus_labels.csv kept=1924 removed=1000
[filter] 011_T415_synchronized_corrected_icml_consensus_labels.csv kept=1855 remove